# 01 — Quickstart & synthetic data

Generate a synthetic banking book and explore it. The generator is a
causal agent-based simulator: labels are *consequences* of latent traits,
so the data has realistic, learnable structure without leakage.

> pragmatiq is an independent implementation inspired by the PRAGMA paper
> (arXiv 2604.08649) and is not affiliated with or endorsed by Revolut.

## Setup

Keep imports and notebook-level constants in one place so later cells focus on
the data workflow rather than environment setup.

In [ ]:
from pathlib import Path

import pyarrow.parquet as pq
from IPython.display import display

from pragmatiq import api

WORK = Path("runs_quickstart")
SYNTH_CONFIG = {
    "n_users": 3000,
    "seed": 0,
    "eval_month_credit": 13,
    "eval_month_short": 19,
}

## Generate the synthetic book

This small run is meant for a laptop. Increase `n_users` when you want a more
realistic book for benchmarking.

In [ ]:
manifest = api.synthesize(
    SYNTH_CONFIG,
    out=WORK / "raw",
    n_workers=4,
    write_report=True,
)

display(manifest["label_prevalence"])

## Events and labels

Events are key–value–time records across four sources
(transaction / app / trading / communication). Labels live in
`labels/*.parquet`, each computed strictly after its eval point.

In [ ]:
events = pq.read_table(WORK / "raw" / "events.parquet").slice(0, 8).to_pandas()

display(events[["user_id", "ts", "source", "fields"]])

In [ ]:
labels = pq.read_table(WORK / "raw" / "labels" / "default_12m.parquet").to_pandas()

print("default_12m rows:", len(labels), "positives:", int(labels.label.sum()))
print("open in a browser:", WORK / "raw" / "realism_report.html")

## Hand-crafted GBDT baseline

The project baseline (`tests/baselines/credit_gbdt.py`) reaches
ROC-AUC ~0.78–0.83 on credit, which is the bar a pragmatiq probe should
approach.